In [1]:
!pip install xarray netCDF4
! pip install matplotlib seaborn

In [2]:
import sys
!{sys.executable} -m pip install scikit-learn

In [3]:
!{sys.executable} -m pip install -r requirements.txt

  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached aiofiles-25.1.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiohttp-3.10.11-cp310-cp310-macosx_11_0_arm64.whl.metadata (7.7 kB)
  Using cached aiohttp_retry-2.8.3-py3-none-any.whl.metadata (8.9 kB)
  Using cached aioice-0.10.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached aiortc-1.14.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached albucore-0.0.33-py3-none-any.whl.metadata (7.8 kB)
  Using cached albumentationsx-2.0.11-py3-none-any.whl.metadata (79 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached anthropic-0.49.0-py3-none-any.whl.metadata (24 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached anywidget-0.9.18-py3-none-a

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd

import os
import imageio.v2 as imageio
from datetime import datetime, timedelta
from scipy import ndimage
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)

In [5]:
clean_ds = xr.open_dataset("full_processed_training_dataset.nc")

# DC = Degree Centrality (or Divergence Centrality?)
# CC = Clustering Coefficient
# BC = Betweenness Centrality
# ID = In-Degree
# OD = Out-Degree
# is_heatwave = binary heatwave mask
# swvl1 = volumetric soil water layer 1
# land_mask = boolean land/sea mask
# u, v = wind components
# z = geopotential height (Z500)

# Quick sanity check on everything
print("=== Dataset Overview ===")
print(f"Time: {clean_ds.time.values[0]} to {clean_ds.time.values[-1]}")
print(f"Total days: {len(clean_ds.time)}")  # expect 2790
print(f"Grid: {len(clean_ds.lat)} lat x {len(clean_ds.lon)} lon")
print(f"Lat: {float(clean_ds.lat.min()):.2f} to {float(clean_ds.lat.max()):.2f}")
print(f"Lon: {float(clean_ds.lon.min()):.2f} to {float(clean_ds.lon.max()):.2f}")

print("\n=== Variable ranges (check for NaNs and outliers) ===")
for var in clean_ds.data_vars:
    data = clean_ds[var].values
    nan_pct = 100 * np.isnan(data).mean()
    print(f"{var:12s}: min={np.nanmin(data):10.3f}, "
          f"max={np.nanmax(data):10.3f}, "
          f"mean={np.nanmean(data):10.3f}, "
          f"NaN%={nan_pct:.1f}%")

print("\n=== Heatwave coverage ===")
hw = clean_ds['is_heatwave'].values
print(f"Heatwave days (any cell): "
      f"{(hw.any(axis=(1,2))).sum()} / {len(clean_ds.time)}")
print(f"Fraction of grid-cell-days that are heatwave: "
      f"{hw.mean()*100:.1f}%")

=== Dataset Overview ===
Time: 1990-06-01T00:00:00.000000000 to 2020-08-30T00:00:00.000000000
Total days: 2667
Grid: 141 lat x 264 lon
Lat: 35.80 to 71.00
Lon: -25.00 to 41.00

=== Variable ranges (check for NaNs and outliers) ===
DC          : min=     0.000, max=     0.191, mean=     0.005, NaN%=0.0%
CC          : min=     0.000, max=     1.000, mean=     0.056, NaN%=0.0%
BC          : min=     0.000, max=     0.000, mean=     0.000, NaN%=0.0%
ID          : min=     0.000, max=  4493.000, mean=    41.565, NaN%=0.0%
OD          : min=     0.000, max=  4460.000, mean=    41.548, NaN%=0.0%
is_heatwave : min=     0.000, max=     1.000, mean=     0.050, NaN%=0.0%
swvl1       : min=    -0.000, max=     0.766, mean=     0.128, NaN%=0.0%
land_mask   : min=     0.000, max=     1.000, mean=     0.465, NaN%=0.0%
u           : min=   -32.276, max=    52.661, mean=     7.104, NaN%=0.0%
v           : min=   -44.702, max=    41.965, mean=     0.543, NaN%=0.0%
z           : min= 50519.625, max= 5881

In [8]:
COEFFS = [
    "BC", "DC", "ID", "OD", "is_heatwave",
    "swvl1", "land_mask", "u", "v", "z"
]

TARGET = "CC_target_next_day"

X_xr = xr.concat(
    [clean_ds[var] for var in COEFFS],
    dim="channel"
).assign_coords(channel=COEFFS)

y_xr = clean_ds[TARGET]

labels = clean_ds["event_label"].values.astype(np.int8)
times = pd.DatetimeIndex(clean_ds.time.values)

print("X_xr:", X_xr.shape)
print("y_xr:", y_xr.shape)
print("labels:", labels.shape)
print("time:", times[0], "to", times[-1])

X_xr: (10, 2667, 141, 264)
y_xr: (2667, 141, 264)
labels: (2667,)
time: 1990-06-01 00:00:00 to 2020-08-30 00:00:00


In [9]:
Xt_vals = X_xr.transpose("time", "channel", "lat", "lon").values.astype(np.float32)
yt_vals = y_xr.transpose("time", "lat", "lon").values.astype(np.float32)

print("Xt_vals:", Xt_vals.shape)
print("yt_vals:", yt_vals.shape)
print("labels:", labels.shape)

Xt_vals: (2667, 10, 141, 264)
yt_vals: (2667, 141, 264)
labels: (2667,)


## Train/val split

In [10]:
val_years = [2016, 2017, 2018, 2019, 2020]

val_mask = times.year.isin(val_years)
train_mask = ~val_mask

Xt_tr = Xt_vals[train_mask]
Xt_val = Xt_vals[val_mask]

yt_tr = yt_vals[train_mask]
yt_val = yt_vals[val_mask]

lbl_tr = labels[train_mask]
lbl_val = labels[val_mask]

tms_tr = times[train_mask]
tms_val = times[val_mask]

print("Train:", Xt_tr.shape, tms_tr[0], "to", tms_tr[-1])
print("Val:", Xt_val.shape, tms_val[0], "to", tms_val[-1])

Train: (2208, 10, 141, 264) 1990-06-01 00:00:00 to 2015-08-31 00:00:00
Val: (459, 10, 141, 264) 2016-06-01 00:00:00 to 2020-08-30 00:00:00


In [11]:
channel_mean = Xt_tr.mean(axis=(0, 2, 3), keepdims=True)
channel_std = Xt_tr.std(axis=(0, 2, 3), keepdims=True)

channel_std[channel_std == 0] = 1.0

Xt_tr = (Xt_tr - channel_mean) / channel_std
Xt_val = (Xt_val - channel_mean) / channel_std

np.save("channel_mean.npy", channel_mean)
np.save("channel_std.npy", channel_std)

print("Normalized.")
print("Train mean after norm:", Xt_tr.mean())
print("Train std after norm:", Xt_tr.std())

Normalized.
Train mean after norm: -3.2112357e-07
Train std after norm: 1.0000209


In [6]:
def choose_coefficients_and_target(ds, coeffs, target):
    return xr.concat([ds[coeff] for coeff in coeffs], dim="channel").assign_coords(channel=list(coeffs)), ds[target]


In [7]:
class SeqDataset(Dataset):
    def __init__(self, X, y, labels, times, seq_len=14):
        self.X = X
        self.y = y
        self.labels = labels
        self.times = times
        self.seq_len = seq_len

        self.indices = []

        for yr in np.unique(times.year):
            yr_idx = np.where(times.year == yr)[0]

            for i in range(len(yr_idx) - seq_len):
                start = yr_idx[i]
                target = yr_idx[i + seq_len]
                self.indices.append((start, target))

        print(f"Created {len(self.indices)} sequences")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start, target = self.indices[idx]

        X_seq = self.X[start:start+self.seq_len]      # (L, C, H, W)
        y_map = self.y[target][None, :, :]            # (1, H, W)
        label = self.labels[target]

        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_map, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )

In [13]:
seq_len = 14
batch_size = 4

train_ds = SeqDataset(Xt_tr, yt_tr, lbl_tr, tms_tr, seq_len=seq_len)
val_ds = SeqDataset(Xt_val, yt_val, lbl_val, tms_val, seq_len=seq_len)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

X_batch, y_batch, label_batch = next(iter(train_loader))

print("X batch:", X_batch.shape)
print("y batch:", y_batch.shape)
print("label batch:", label_batch.shape)

Created 1872 sequences
Created 389 sequences
X batch: torch.Size([4, 14, 10, 141, 264])
y batch: torch.Size([4, 1, 141, 264])
label batch: torch.Size([4])


/opt/anaconda3/envs/aml_py310/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## Model

In [14]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2
        self.hidden_dim = hidden_dim

        self.conv = nn.Conv2d(
            input_dim + hidden_dim,
            4 * hidden_dim,
            kernel_size=kernel_size,
            padding=padding
        )

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)

        i, f, o, g = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next


class MultiHeadConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, kernel_size=3, n_classes=2):
        super().__init__()

        self.cell = ConvLSTMCell(input_dim, hidden_dim, kernel_size)

        # Head 1: spatial CC regression
        self.cc_head = nn.Conv2d(hidden_dim, 1, kernel_size=1)

        # Head 2: standing vs propagating classification
        self.class_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        B, L, C, H, W = x.shape

        h = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)
        c = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)

        for t in range(L):
            h, c = self.cell(x[:, t], h, c)

        cc_pred = torch.sigmoid(self.cc_head(h))   # (B, 1, H, W)
        class_logits = self.class_head(h)          # (B, 2)

        return cc_pred, class_logits

In [15]:
def compute_loss(
    cc_pred,
    class_logits,
    y_cc,
    y_class,
    w_cc=1.0,
    w_class=1.0,
    class_weights=None,
    device="cpu"
):
    losses = {}

    # CC map regression
    losses["cc"] = nn.MSELoss()(cc_pred, y_cc)

    # Only use samples where event_label is 0 or 1
    valid_mask = y_class >= 0

    if valid_mask.sum() > 0:
        logits_valid = class_logits[valid_mask]
        y_valid = y_class[valid_mask].long()

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        losses["class"] = criterion(logits_valid, y_valid)

    else:
        losses["class"] = torch.tensor(0.0, device=device)

    total_loss = w_cc * losses["cc"] + w_class * losses["class"]

    return total_loss, losses

## Initialize model

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = MultiHeadConvLSTM(
    input_dim=len(COEFFS),
    hidden_dim=32,
    kernel_size=3,
    n_classes=2
).to(device)

valid_train_labels = lbl_tr[lbl_tr >= 0]

n_standing = np.sum(valid_train_labels == 0)
n_propagating = np.sum(valid_train_labels == 1)

print("Standing labels:", n_standing)
print("Propagating labels:", n_propagating)

# Class weights for imbalance
total = n_standing + n_propagating

class_weights = torch.tensor(
    [
        total / max(2 * n_standing, 1),
        total / max(2 * n_propagating, 1)
    ],
    dtype=torch.float32,
    device=device
)

print("Class weights:", class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=5,
    factor=0.5
)

Device: cpu
Standing labels: 826
Propagating labels: 169
Class weights: tensor([0.6023, 2.9438])


In [ ]:
n_epochs = 50

w_cc = 1.0
w_class = 1.0

history = {
    "train_total": [],
    "val_total": [],
    "train_cc": [],
    "val_cc": [],
    "train_class": [],
    "val_class": []
}

best_val_loss = np.inf

for epoch in range(n_epochs):

    model.train()

    train_total = 0.0
    train_cc = 0.0
    train_class = 0.0

    for X, y_cc, y_class in train_loader:
        X = X.to(device, non_blocking=True)
        y_cc = y_cc.to(device, non_blocking=True)
        y_class = y_class.to(device, non_blocking=True)

        optimizer.zero_grad()

        cc_pred, class_logits = model(X)

        loss, losses = compute_loss(
            cc_pred,
            class_logits,
            y_cc,
            y_class,
            w_cc=w_cc,
            w_class=w_class,
            class_weights=class_weights,
            device=device
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_total += loss.item()
        train_cc += losses["cc"].item()
        train_class += losses["class"].item()

    train_total /= len(train_loader)
    train_cc /= len(train_loader)
    train_class /= len(train_loader)

    model.eval()

    val_total = 0.0
    val_cc = 0.0
    val_class = 0.0

    with torch.no_grad():
        for X, y_cc, y_class in val_loader:
            X = X.to(device, non_blocking=True)
            y_cc = y_cc.to(device, non_blocking=True)
            y_class = y_class.to(device, non_blocking=True)

            cc_pred, class_logits = model(X)

            loss, losses = compute_loss(
                cc_pred,
                class_logits,
                y_cc,
                y_class,
                w_cc=w_cc,
                w_class=w_class,
                class_weights=class_weights,
                device=device
            )

            val_total += loss.item()
            val_cc += losses["cc"].item()
            val_class += losses["class"].item()

    val_total /= len(val_loader)
    val_cc /= len(val_loader)
    val_class /= len(val_loader)

    scheduler.step(val_total)

    history["train_total"].append(train_total)
    history["val_total"].append(val_total)
    history["train_cc"].append(train_cc)
    history["val_cc"].append(val_cc)
    history["train_class"].append(train_class)
    history["val_class"].append(val_class)

    if val_total < best_val_loss:
        best_val_loss = val_total
        torch.save(model.state_dict(), "best_multihead_convlstm.pt")

    print(
        f"Epoch {epoch+1:03d}/{n_epochs} | "
        f"train={train_total:.4f} | val={val_total:.4f} | "
        f"CC val={val_cc:.4f} | class val={val_class:.4f}"
    )

print("Best val loss:", best_val_loss)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_total"], label="train total")
plt.plot(history["val_total"], label="val total")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Total multi-head loss")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_cc"], label="train CC")
plt.plot(history["val_cc"], label="val CC")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("CC regression loss")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_class"], label="train classification")
plt.plot(history["val_class"], label="val classification")
plt.xlabel("Epoch")
plt.ylabel("CrossEntropy")
plt.legend()
plt.title("Standing vs propagating classification loss")
plt.show()

In [ ]:
model.load_state_dict(torch.load("best_multihead_convlstm.pt", map_location=device))
model.eval()

all_cc_pred = []
all_cc_true = []

all_class_true = []
all_class_pred = []
all_class_prob = []

with torch.no_grad():
    for X, y_cc, y_class in val_loader:
        X = X.to(device, non_blocking=True)

        cc_pred, class_logits = model(X)

        all_cc_pred.append(cc_pred.cpu().numpy())
        all_cc_true.append(y_cc.numpy())

        probs = torch.softmax(class_logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        y_class_np = y_class.numpy()
        valid_mask = y_class_np >= 0

        all_class_true.extend(y_class_np[valid_mask].tolist())
        all_class_pred.extend(preds[valid_mask].tolist())
        all_class_prob.extend(probs[valid_mask, 1].tolist())

cc_pred = np.concatenate(all_cc_pred, axis=0)
cc_true = np.concatenate(all_cc_true, axis=0)

mae = np.mean(np.abs(cc_pred - cc_true))
rmse = np.sqrt(np.mean((cc_pred - cc_true) ** 2))

ss_res = np.sum((cc_true - cc_pred) ** 2)
ss_tot = np.sum((cc_true - cc_true.mean()) ** 2)

r2 = 1 - ss_res / (ss_tot + 1e-8)

print("=== CC regression ===")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

y_true = np.array(all_class_true)
y_pred = np.array(all_class_pred)

print("\n=== Standing vs propagating classification ===")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))

cm = confusion_matrix(y_true, y_pred)

ConfusionMatrixDisplay(
    cm,
    display_labels=["standing", "propagating"]
).plot(colorbar=False)

plt.title("Standing vs propagating confusion matrix")
plt.show()

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "coeffs": COEFFS,
    "target": TARGET,
    "seq_len": seq_len,
    "channel_mean": channel_mean,
    "channel_std": channel_std,
    "hidden_dim": 32,
    "class_names": ["standing", "propagating"],
    "class_weights": class_weights.detach().cpu().numpy(),
}, "multihead_convlstm_colab_ready.pt")

print("Saved: multihead_convlstm_colab_ready.pt")

## later stuff

In [ ]:
# Confirm clean_ds_filtered has no 1997/1999
clean_years = [yr for yr in range(1990, 2021) if yr not in [1997, 1999]]
clean_ds_filtered = clean_ds.sel(
    time=clean_ds.time.dt.year.isin(clean_years)
)

years_in_filtered = np.unique(pd.DatetimeIndex(clean_ds_filtered.time.values).year)
print(f"Years in filtered ds: {years_in_filtered}")
print(f"N years: {len(years_in_filtered)}  ← expect 29")
print(f"N days:  {len(clean_ds_filtered.time)}  ← expect 2668")
print(f"1997 present: {1997 in years_in_filtered}  ← expect False")
print(f"1999 present: {1999 in years_in_filtered}  ← expect False")

Years in filtered ds: [1990 1991 1992 1993 1994 1995 1996 1998 2000 2001 2002 2003 2004 2005
 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019
 2020]
N years: 29  ← expect 29
N days:  2668  ← expect 2668
1997 present: False  ← expect False
1999 present: False  ← expect False


In [ ]:
def choose_coefficients_and_target(ds, coeffs, target):
    return xr.concat([ds[coeff] for coeff in coeffs], dim="channel").assign_coords(channel=list(coeffs)), ds[target]


In [ ]:
COEFFS = ["BC", "DC", "ID", "OD", "is_heatwave",
          "swvl1", "land_mask", "u", "v", "z"]
TARGET = "CC"

X_xr, y_xr = choose_coefficients_and_target(clean_ds_filtered, COEFFS, TARGET)
y_xr_shifted = y_xr.shift(time=-1)
X_xr_trim    = X_xr.isel(time=slice(None, -1))
y_xr_trim    = y_xr_shifted.isel(time=slice(None, -1))

print(f"X_xr shape:  {X_xr_trim.shape}   ← expect (10, 2667, 141, 264)")
print(f"y_xr shape:  {y_xr_trim.shape}   ← expect (2667, 141, 264)")

# Check time axes match
print(f"\nX time[0]:  {X_xr_trim.time.values[0]}")
print(f"y time[0]:  {y_xr_trim.time.values[0]}")
print(f"X time[-1]: {X_xr_trim.time.values[-1]}")
print(f"y time[-1]: {y_xr_trim.time.values[-1]}")
# Both should be identical — they're aligned to the same time axis

X_xr shape:  (10, 2667, 141, 264)   ← expect (10, 2667, 141, 264)
y_xr shape:  (2667, 141, 264)   ← expect (2667, 141, 264)

X time[0]:  1990-06-01T00:00:00.000000000
y time[0]:  1990-06-01T00:00:00.000000000
X time[-1]: 2020-08-30T00:00:00.000000000
y time[-1]: 2020-08-30T00:00:00.000000000


## Checks before running

In [ ]:
div_aligned    = div_clean[:-1]        # (2667, 141, 264)
labels_aligned = daily_labels[:-1]     # (2667,)
times_aligned  = times_clean[:-1]      # (2667,)

print(f"div_aligned shape:    {div_aligned.shape}    ← expect (2667, 141, 264)")
print(f"labels_aligned shape: {labels_aligned.shape} ← expect (2667,)")
print(f"times_aligned shape:  {times_aligned.shape}  ← expect (2667,)")

# Check time axes all agree
xr_times = pd.DatetimeIndex(X_xr_trim.time.values)
print(f"\nFirst date — X_xr: {xr_times[0].date()}  "
      f"times_aligned: {pd.Timestamp(times_aligned[0]).date()}")
print(f"Last date  — X_xr: {xr_times[-1].date()}  "
      f"times_aligned: {pd.Timestamp(times_aligned[-1]).date()}")
# Must match exactly

div_aligned shape:    (2667, 141, 264)    ← expect (2667, 141, 264)
labels_aligned shape: (2667,) ← expect (2667,)
times_aligned shape:  (2667,)  ← expect (2667,)

First date — X_xr: 1990-06-01  times_aligned: 1990-06-01
Last date  — X_xr: 2020-08-30  times_aligned: 2020-08-30


In [ ]:
# ── dry run: 2003 ──────────────────────────────────────────────────────

test_yr = 2003
yr_mask = times_aligned.year == test_yr

# Use yr_mask (numpy boolean) for X instead of .sel(time=str(yr))
# This is also how make_sequences_extended should index X
Xt = X_xr_trim.transpose("time", "channel", "lat", "lon")
yt = y_xr_trim.transpose("time", "lat", "lon")

Xv  = Xt.values[yr_mask]    # (92, 10, 141, 264)  ← index with numpy mask
yv  = yt.sel(time=str(test_yr)).values             # (92, 141, 264)
div = div_aligned[yr_mask]                         # (92, 141, 264)
lbl = labels_aligned[yr_mask]                      # (92,)

print(f"=== Dry run: {test_yr} ===")
print(f"Xv:  {Xv.shape}   ← expect (92, 10, 141, 264)")
print(f"yv:  {yv.shape}   ← expect (92, 141, 264)")
print(f"div: {div.shape}  ← expect (92, 141, 264)")
print(f"lbl: {lbl.shape}  ← expect (92,)")

seq_len = 14
x_sample  = Xv[0:seq_len]
y_sample  = yv[seq_len]
yp_sample = lbl[seq_len]
yd_sample = div[seq_len]

print(f"\nSingle sequence shapes:")
print(f"  x_sample:  {x_sample.shape}   ← expect (14, 10, 141, 264)")
print(f"  y_sample:  {y_sample.shape}   ← expect (141, 264)")
print(f"  yp_sample: {yp_sample}        ← expect -1, 0, or 1")
print(f"  yd_sample: {yd_sample.shape}  ← expect (141, 264)")

=== Dry run: 2003 ===
Xv:  (92, 10, 141, 264)   ← expect (92, 10, 141, 264)
yv:  (92, 141, 264)   ← expect (92, 141, 264)
div: (92, 141, 264)  ← expect (92, 141, 264)
lbl: (92,)  ← expect (92,)

Single sequence shapes:
  x_sample:  (14, 10, 141, 264)   ← expect (14, 10, 141, 264)
  y_sample:  (141, 264)   ← expect (141, 264)
  yp_sample: 0        ← expect -1, 0, or 1
  yd_sample: (141, 264)  ← expect (141, 264)


In [ ]:
model_test = ConvLSTM(input_dim=10, hidden_dim=32,
                      kernel_size=3, use_new_heads=True)
model_test.eval()

# Fake batch: 2 samples, 14 timesteps, 10 channels, 141×264
x_fake = torch.randn(2, 14, 10, 141, 264)

with torch.no_grad():
    out_main, out_prop, out_div = model_test(x_fake)

print(f"out_main shape: {out_main.shape}  ← expect (2, 1, 141, 264)")
print(f"out_prop shape: {out_prop.shape}  ← expect (2, 1)")
print(f"out_div  shape: {out_div.shape}   ← expect (2, 1, 141, 264)")
print(f"\nout_main range: {out_main.min():.3f} to {out_main.max():.3f}  "
      f"← expect [0,1] for CC_regression")
print(f"out_prop range: {out_prop.min():.3f} to {out_prop.max():.3f}  "
      f"← unbounded logits")
print(f"out_div  range: {out_div.min():.3f} to {out_div.max():.3f}   "
      f"← unbounded")

print(f"\nModel parameters: "
      f"{sum(p.numel() for p in model_test.parameters()):,}")

out_main shape: torch.Size([2, 1, 141, 264])  ← expect (2, 1, 141, 264)
out_prop shape: torch.Size([2, 1])  ← expect (2, 1)
out_div  shape: torch.Size([2, 1, 141, 264])   ← expect (2, 1, 141, 264)

out_main range: 0.475 to 0.575  ← expect [0,1] for CC_regression
out_prop range: -0.092 to -0.092  ← unbounded logits
out_div  range: -0.164 to 0.255   ← unbounded

Model parameters: 49,667


## 5. Run the ConvLSTM model

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# TASK TYPE — set this before running
# ─────────────────────────────────────────────────────────────────────────────
TASK_TYPE = "CC_regression"   # existing head: "binary" | "regression" | "CC_regression"
# New heads are always active when USE_NEW_HEADS = True
USE_NEW_HEADS = True


# ─────────────────────────────────────────────────────────────────────────────
# DATASET — extended to carry prop/standing label + divergence target
# ─────────────────────────────────────────────────────────────────────────────

class NumpySeqDataset(Dataset):
    """
    Wraps numpy arrays into a PyTorch Dataset.

    Args:
        X       : (N, L, C, H, W)  float32 — input sequences
        y       : (N, 1, H, W)     float32 — existing gridded target (CC / HW / etc.)
        y_prop  : (N,)             int8    — prop/standing label (-1 = no event,
                                             0 = standing, 1 = propagating)
        y_div   : (N, 1, H, W)    float32 — divergence field at target day
                  Pass None for either new target to skip that head.
    """
    def __init__(self, X, y, y_prop=None, y_div=None):
        self.X      = torch.from_numpy(X).float()
        self.y      = torch.from_numpy(y).float()
        self.y_prop = (torch.from_numpy(y_prop.astype(np.int64))
                       if y_prop is not None else None)
        self.y_div  = (torch.from_numpy(y_div).float()
                       if y_div is not None else None)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        out = {"X": self.X[i], "y": self.y[i]}
        if self.y_prop is not None:
            out["y_prop"] = self.y_prop[i]
        if self.y_div is not None:
            out["y_div"]  = self.y_div[i]
        return out

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────────────

class ConvLSTMCell(nn.Module):
    """Unchanged from original."""
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size=kernel_size, padding=padding)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates    = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class ConvLSTM(nn.Module):
    """
    Multi-task ConvLSTM.

    Heads:
      head_main  — original gridded output (CC / HW / divergence regression)
      head_prop  — NEW: scalar binary classifier (propagating vs standing)
                   outputs a single logit per sample → BCEWithLogitsLoss
      head_div   — NEW: gridded divergence regression (same shape as head_main)
                   outputs unbounded float map → MSELoss

    Only head_main is active when USE_NEW_HEADS = False (backward compatible).
    """
    def __init__(self, input_dim, hidden_dim=32, kernel_size=3,
                 use_new_heads=True):
        super().__init__()
        self.use_new_heads = use_new_heads

        # ── Shared encoder (unchanged) ────────────────────────────────────
        self.cell = ConvLSTMCell(input_dim, hidden_dim, kernel_size)

        # ── Head 1: original gridded output (unchanged) ───────────────────
        self.head_main = nn.Conv2d(hidden_dim, 1, kernel_size=1)

        if use_new_heads:
            # ── Head 2: propagating/standing scalar classifier ────────────
            # Global average pool → flatten → linear → single logit
            # GAP collapses (H, W) → scalar per channel, preserving spatial
            # summary without adding spatial parameters
            self.head_prop = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),   # (B, hidden_dim, 1, 1)
                nn.Flatten(),              # (B, hidden_dim)
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(32, 1)           # (B, 1) — raw logit
            )

            # ── Head 3: divergence field regression ───────────────────────
            # Same architecture as head_main but predicts divergence map
            self.head_div = nn.Conv2d(hidden_dim, 1, kernel_size=1)

    def forward(self, x):
        """
        Args:
            x : (B, L, C, H, W)
        Returns:
            out_main  : (B, 1, H, W)   — existing target (CC / HW / etc.)
            out_prop  : (B, 1)         — prop/standing logit (if use_new_heads)
            out_div   : (B, 1, H, W)   — divergence map     (if use_new_heads)
        """
        B, L, C, H, W = x.shape
        h = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)
        c = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)

        for t in range(L):
            h, c = self.cell(x[:, t], h, c)

        # ── Main head (unchanged logic) ───────────────────────────────────
        out_main = self.head_main(h)   # (B, 1, H, W)

        if TASK_TYPE == "binary":
            pass                        # raw logits
        elif TASK_TYPE == "CC_regression":
            out_main = torch.sigmoid(out_main)
        # "regression" → unbounded, no activation

        if not self.use_new_heads:
            return out_main

        # ── New heads ─────────────────────────────────────────────────────
        out_prop = self.head_prop(h)   # (B, 1)
        out_div  = self.head_div(h)    # (B, 1, H, W) — unbounded

        return out_main, out_prop, out_div


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SEQUENCE BUILDER — extended to include prop label + divergence target
# ─────────────────────────────────────────────────────────────────────────────

def make_sequences_extended(X_xr, y_xr, div_np, daily_labels_np,
                             times_clean, seq_len=14):

    # Transpose once outside the loop
    Xt_vals = X_xr.transpose("time", "channel", "lat", "lon").values  # (T, C, H, W)
    yt      = y_xr.transpose("time", "lat", "lon")                     # keep as xarray for .sel

    years = np.unique(times_clean.year)
    X_list, y_list, yp_list, yd_list, d_list = [], [], [], [], []

    for yr in years:
        yr_mask = times_clean.year == yr

        Xv  = Xt_vals[yr_mask]                          # (92, C, H, W)  ← numpy mask
        yv  = yt.sel(time=str(yr)).values               # (92, H, W)
        div = div_np[yr_mask]                           # (92, H, W)
        lbl = daily_labels_np[yr_mask]                  # (92,)
        tms = times_clean[yr_mask]

        T = Xv.shape[0]
        if T <= seq_len:
            continue

        for i in range(T - seq_len):
            target_idx = i + seq_len
            X_list.append(Xv[i:i + seq_len])
            y_list.append(yv[target_idx])
            yp_list.append(lbl[target_idx])
            yd_list.append(div[target_idx])
            d_list.append(tms[target_idx])

    X_seq   = np.stack(X_list).astype(np.float32)
    y_seq   = np.stack(y_list).astype(np.float32)[:, None]
    y_prop  = np.array(yp_list, dtype=np.int8)
    y_div   = np.stack(yd_list).astype(np.float32)[:, None]
    y_dates = np.array(d_list)

    print(f"Sequences built: {X_seq.shape[0]}")
    print(f"  X_seq  : {X_seq.shape}   ← expect (N, {seq_len}, 10, 141, 264)")
    print(f"  y_seq  : {y_seq.shape}   ← expect (N, 1, 141, 264)")
    print(f"  y_prop : {y_prop.shape}  "
          f"(standing={(y_prop==0).sum()}, "
          f"prop={(y_prop==1).sum()}, "
          f"no-event={(y_prop==-1).sum()})")
    print(f"  y_div  : {y_div.shape}   ← expect (N, 1, 141, 264)")

    return X_seq, y_seq, y_prop, y_div, y_dates

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOSS — multi-task with class weighting for imbalanced prop/standing
# ─────────────────────────────────────────────────────────────────────────────

def compute_loss(out_main, out_prop, out_div, batch,
                 w_main=1.0, w_prop=1.0, w_div=0.5,
                 prop_pos_weight=None, device='cpu'):
    """
    Combined loss across all three heads.

    w_main, w_prop, w_div : loss weights — tune these
    prop_pos_weight       : scalar tensor to upweight propagating class
                            = n_standing / n_propagating ≈ 964/205 ≈ 4.7
    """
    losses = {}

    # ── Head 1: existing loss (unchanged) ────────────────────────────────
    y_main = batch["y"].to(device)
    if TASK_TYPE == "binary":
        losses["main"] = nn.BCEWithLogitsLoss()(out_main, y_main)
    else:
        losses["main"] = nn.MSELoss()(out_main, y_main)

    # ── Head 2: prop/standing loss ────────────────────────────────────────
    # Only compute on samples that have a valid HW label (label >= 0)
    y_prop_all = batch["y_prop"].to(device)          # (B,)  values: -1, 0, 1
    hw_mask    = y_prop_all >= 0                     # (B,)  True = has HW label

    if hw_mask.sum() > 0:
        y_prop_hw  = y_prop_all[hw_mask].float()     # (B',) values: 0 or 1
        out_prop_hw = out_prop[hw_mask]              # (B', 1)

        if prop_pos_weight is not None:
            pw = torch.tensor([prop_pos_weight],
                              dtype=torch.float32, device=device)
            criterion_prop = nn.BCEWithLogitsLoss(pos_weight=pw)
        else:
            criterion_prop = nn.BCEWithLogitsLoss()

        losses["prop"] = criterion_prop(
            out_prop_hw.squeeze(1), y_prop_hw
        )
    else:
        losses["prop"] = torch.tensor(0.0, device=device)

    # ── Head 3: divergence loss ───────────────────────────────────────────
    y_div = batch["y_div"].to(device)                # (B, 1, H, W)
    losses["div"] = nn.MSELoss()(out_div, y_div)

    # ── Combined ──────────────────────────────────────────────────────────
    total = w_main * losses["main"] \
          + w_prop * losses["prop"] \
          + w_div  * losses["div"]

    return total, losses


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train_model(model, train_loader, val_loader,
                n_epochs=50, lr=1e-3,
                prop_pos_weight=4.7,   # n_standing / n_propagating ≈ 964/205
                w_main=1.0, w_prop=1.0, w_div=0.5,
                device='cpu'):
    """
    prop_pos_weight: upweights propagating class to correct for imbalance.
    Set to n_standing / n_propagating from your daily label counts.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=True
    )
    model.to(device)

    history = {"train_total": [], "val_total": [],
               "train_main": [], "val_main": [],
               "train_prop": [], "val_prop": [],
               "train_div":  [], "val_div":  []}

    for epoch in range(n_epochs):
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_losses = {"total": 0, "main": 0, "prop": 0, "div": 0}

        for batch in train_loader:
            optimizer.zero_grad()
            x = batch["X"].to(device)

            out_main, out_prop, out_div = model(x)

            loss, losses = compute_loss(
                out_main, out_prop, out_div, batch,
                w_main=w_main, w_prop=w_prop, w_div=w_div,
                prop_pos_weight=prop_pos_weight,
                device=device
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_losses["total"] += loss.item()
            for k in ["main", "prop", "div"]:
                epoch_losses[k] += losses[k].item()

        n = len(train_loader)
        for k in epoch_losses:
            history[f"train_{k}"].append(epoch_losses[k] / n)

        # ── Validate ──────────────────────────────────────────────────────
        model.eval()
        val_losses = {"total": 0, "main": 0, "prop": 0, "div": 0}

        with torch.no_grad():
            for batch in val_loader:
                x = batch["X"].to(device)
                out_main, out_prop, out_div = model(x)
                loss, losses = compute_loss(
                    out_main, out_prop, out_div, batch,
                    w_main=w_main, w_prop=w_prop, w_div=w_div,
                    prop_pos_weight=prop_pos_weight,
                    device=device
                )
                val_losses["total"] += loss.item()
                for k in ["main", "prop", "div"]:
                    val_losses[k] += losses[k].item()

        nv = len(val_loader)
        for k in val_losses:
            history[f"val_{k}"].append(val_losses[k] / nv)

        scheduler.step(history["val_total"][-1])

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}/{n_epochs} | "
                  f"train {history['train_total'][-1]:.4f} "
                  f"(main {history['train_main'][-1]:.4f} "
                  f"prop {history['train_prop'][-1]:.4f} "
                  f"div {history['train_div'][-1]:.4f}) | "
                  f"val {history['val_total'][-1]:.4f} "
                  f"(main {history['val_main'][-1]:.4f} "
                  f"prop {history['val_prop'][-1]:.4f} "
                  f"div {history['val_div'][-1]:.4f})")

    return model, history

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# WIRING — run this block to build datasets and start training
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── 1. Build sequences ────────────────────────────────────────────────
    # X and y come from your existing choose_coefficients_and_target call
    # but use the CLEAN dataset (1997/1999 dropped) and shift y by -1 as before

    COEFFS = ["BC", "DC", "ID", "OD", "is_heatwave",
              "swvl1", "land_mask", "u", "v", "z"]
    TARGET = "CC"

    # Rebuild X/y on clean_ds filtered to clean years
    clean_years = [yr for yr in range(1990, 2021) if yr not in [1997, 1999]]
    clean_ds_filtered = clean_ds.sel(
        time=clean_ds.time.dt.year.isin(clean_years)
    )

    X_xr, y_xr = choose_coefficients_and_target(
        clean_ds_filtered, COEFFS, TARGET
    )
    y_xr = y_xr.shift(time=-1)
    X_xr = X_xr.isel(time=slice(None, -1))
    y_xr = y_xr.isel(time=slice(None, -1))

    # Align divergence and labels (also drop last step to match y shift)
    div_aligned    = div_clean[:-1]           # (2667, 141, 264)
    labels_aligned = daily_labels[:-1]        # (2667,)
    times_aligned  = times_clean[:-1]         # (2667,)

    seq_len = 14
    X_seq, y_seq, y_prop, y_div, y_dates = make_sequences_extended(
        X_xr, y_xr, div_aligned, labels_aligned,
        times_aligned, seq_len=seq_len
    )

    # ── 2. Train/val split — last 5 years as validation ──────────────────
    val_years  = [2016, 2017, 2018, 2019, 2020]
    val_dates  = pd.DatetimeIndex(y_dates)
    val_mask   = val_dates.year.isin(val_years)
    train_mask = ~val_mask

    def split(arr, mask): return arr[mask], arr[~mask]

    X_tr, X_val     = X_seq[train_mask],  X_seq[val_mask]
    y_tr, y_val     = y_seq[train_mask],  y_seq[val_mask]
    yp_tr, yp_val   = y_prop[train_mask], y_prop[val_mask]
    yd_tr, yd_val   = y_div[train_mask],  y_div[val_mask]

    print(f"Train: {X_tr.shape[0]} samples | Val: {X_val.shape[0]} samples")
    print(f"Train prop/stand: {(yp_tr==1).sum()} / {(yp_tr==0).sum()}")
    print(f"Val   prop/stand: {(yp_val==1).sum()} / {(yp_val==0).sum()}")

    # ── 3. DataLoaders ────────────────────────────────────────────────────
    train_ds = NumpySeqDataset(X_tr, y_tr, yp_tr, yd_tr)
    val_ds   = NumpySeqDataset(X_val, y_val, yp_val, yd_val)

    train_loader = DataLoader(train_ds, batch_size=16,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=16,
                              shuffle=False, num_workers=0)

    # ── 4. Model ──────────────────────────────────────────────────────────
    device     = "cuda" if torch.cuda.is_available() else "cpu"
    input_dim  = len(COEFFS)   # 10 channels

    model = ConvLSTM(
        input_dim    = input_dim,
        hidden_dim   = 32,
        kernel_size  = 3,
        use_new_heads= True
    )

    print(f"\nModel parameters: "
          f"{sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device}")

    # ── 5. Train ──────────────────────────────────────────────────────────
    # prop_pos_weight = n_standing / n_propagating from your daily label array
    prop_pos_weight = (yp_tr == 0).sum() / max((yp_tr == 1).sum(), 1)
    print(f"prop_pos_weight: {prop_pos_weight:.2f}")

    model, history = train_model(
        model, train_loader, val_loader,
        n_epochs        = 50,
        lr              = 1e-3,
        prop_pos_weight = prop_pos_weight,
        w_main          = 1.0,
        w_prop          = 1.0,
        w_div           = 0.5,
        device          = device
    )

    torch.save(model.state_dict(), "convlstm_multitask.pt")
    print("Model saved: convlstm_multitask.pt")

In [ ]:
seq_len = 14

X_seq, y_seq, y_prop, y_div, y_dates = make_sequences_extended(
    X_xr_trim, y_xr_trim,
    div_aligned, labels_aligned,
    times_aligned, seq_len=seq_len
)

: 

In [ ]:
from scipy.interpolate import interp1d

def extract_pathway_features(events, n_points=10):
    """
    Interpolates each propagating event track to n_points
    so all tracks have the same length for clustering.
    Returns feature matrix of shape (n_prop_events, n_points*2)
    """
    prop_events = [ev for ev in events if ev['label'] == 1]
    features    = []
    valid_events = []

    for ev in prop_events:
        c = ev['centers']   # (T, 2) where T <= 10
        if len(c) < 3:      # need at least 3 points to interpolate
            continue

        t_orig = np.linspace(0, 1, len(c))
        t_new  = np.linspace(0, 1, n_points)

        lat_interp = interp1d(t_orig, c[:, 0], kind='linear')(t_new)
        lon_interp = interp1d(t_orig, c[:, 1], kind='linear')(t_new)

        features.append(np.concatenate([lat_interp, lon_interp]))
        valid_events.append(ev)

    return np.array(features), valid_events


# ── Cluster propagating tracks into pathways ─────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features, prop_events = extract_pathway_features(events, n_points=10)
print(f"Propagating events with full tracks: {len(prop_events)}")

# Try 2 and 3 clusters
fig, axes = plt.subplots(1, 2, figsize=(18, 5),
                          subplot_kw={'projection': ccrs.PlateCarree()})

cluster_colors = ['tomato', 'steelblue', 'forestgreen']

for ax, n_clusters in zip(axes, [2, 3]):
    X_scaled = StandardScaler().fit_transform(features)
    km       = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    pathway_labels = km.fit_predict(X_scaled)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4)
    ax.add_feature(cfeature.LAND,      facecolor='lightgray', alpha=0.3)
    ax.set_extent([-25, 41, 35, 71])
    ax.set_title(f'{n_clusters} pathway clusters', fontsize=11)

    for ev, lab in zip(prop_events, pathway_labels):
        c   = ev['centers']
        col = cluster_colors[lab]
        ax.plot(c[:, 1], c[:, 0], '-o', color=col,
                linewidth=1.2, markersize=3, alpha=0.6,
                transform=ccrs.PlateCarree())
        # Mark onset with larger dot
        ax.plot(c[0, 1], c[0, 0], 'o', color=col,
                markersize=6, alpha=0.9,
                transform=ccrs.PlateCarree())

    # Plot cluster mean tracks
    for k in range(n_clusters):
        mask     = pathway_labels == k
        mean_f   = features[mask].mean(axis=0)
        mean_lat = mean_f[:10]
        mean_lon = mean_f[10:]
        ax.plot(mean_lon, mean_lat, '-', color=cluster_colors[k],
                linewidth=3, alpha=1.0, transform=ccrs.PlateCarree(),
                label=f'Pathway {k+1} (n={mask.sum()})')
        ax.plot(mean_lon[0], mean_lat[0], '*', color=cluster_colors[k],
                markersize=14, transform=ccrs.PlateCarree())

    ax.legend(loc='lower left', fontsize=9)

plt.suptitle('European heatwave propagation pathways', fontsize=13)
plt.tight_layout()
plt.savefig('heatwave_pathways.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: heatwave_pathways.png")

NameError: name 'events' is not defined